In [4]:
%reload_ext autoreload
%autoreload 2

# Imports

In [5]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

## Unit tests

In [7]:
from kret_np_pd.notebooks import test_get_nearby_rows as tgnr

In [9]:
results = tgnr.run_all()

[PASS] ex1: hard match (gameId, teamId)
[PASS] ex2: soft match {time: (5, backward)}
[PASS] ex3: soft {time:(2,forward), dist:10 (default both)}, and
[PASS] ex4: hard match (gameId, period)
[PASS] ex5: hard playId AND soft {time:(3,backward), dist:5 (default both)}
[PASS] ex6: hard gameId AND soft {time:2 (both)}
[PASS] empty filt → all False
[PASS] no hard, no soft → returns filt unchanged
[PASS] hard_match_apply=or
[PASS] soft_match_apply=or
[PASS] join_hard_soft=or includes row passing only soft
[PASS] join_hard_soft=and excludes row passing only soft
[PASS] mixed directions: t1 forward 1, t2 default both 1
[PASS] per-col direction overrides default (default=forward, col=backward)
[PASS] datetime soft match: within 60s of seed timestamp
[PASS] multiple seeds: union of per-seed neighborhoods

16/16 passed


### Inspect a single case

In [ ]:
case = tgnr.case_ex5_hard_play_soft_time_dist()
actual, passed, err = tgnr.run_one(case)
print(f"{case.name}: {'PASS' if passed else 'FAIL'}")
if err:
    print(err)
viz = case.df.assign(
    seed=np.asarray(case.filt).astype(bool),
    expected=case.expected,
    actual=actual if actual is not None else [None] * len(case.df),
)
dtt(viz, n=-1)

# NBA Data

## Load Data

In [ ]:
df = pd.read_parquet("./scratch-nbastatsv3.parquet")

In [16]:
df.actionType.value_counts()

actionType
Rebound                                     2419998
Missed Shot                                 2133422
Made Shot                                   1749170
Free Throw                                  1166817
Foul                                        1053875
Substitution                                 945883
Turnover                                     693023
Timeout                                      320396
period                                       194452
Jump Ball                                     41574
Violation                                     40149
Instant Replay                                18100
Ejection                                       1569
Timeout                                          88
Turnover                                         77
Made Shot                                        49
Missed Shot                                      39
Ejection                                         17
Violation                                         4
N

In [13]:
UKS_NP_PD.dtt(df, n=5)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,season,season_type,seconds_remaining,seconds_elapsed,game_seconds_elapsed,pc_seconds,IsPlayoff,shotValue,game_date,game_date_ffill
,int64,str,int64,int64,str,int64,str,str,int64,int64,int64,str,int64,float64,float64,int64,str,str,str,str,int64,int64,int64,int64,str,float64,float64,float64,int32,bool,float64,datetime64[ns],datetime64[ns]
99948,366,PT03M43.00S,3,1610612754,IND,696,Best,T. Best,-117,76,14,Made,1,89.000,69.000,158,v,Best 14' Jump Shot (5 PTS),Made Shot,Jump Shot,0,373,20000206,2000,rg,223.000,497.000,1937.000,223,False,0.000,NaT,NaT
601050,42,PT06M47.00S,1,1610612762,UTA,731,Ostertag,G. Ostertag,0,0,0,NaN,0,0.000,0.000,0,h,Ostertag S.FOUL (P1.T3),Foul,Shooting,0,45,20100068,2001,rg,407.000,313.000,313.000,407,False,0.000,NaT,NaT
2573570,447,PT00M22.90S,4,1610612750,MIN,84,Sprewell,L. Sprewell,0,0,0,NaN,0,0.000,0.000,0,v,Sprewell P.FOUL (P3.PN),Foul,Personal,0,447,20400662,2004,rg,22.900,697.100,2857.100,23,False,0.000,NaT,NaT
6683547,378,PT09M19.00S,4,1610612753,ORL,2047,Richardson,Q. Richardson,223,108,25,Made,1,93.000,70.000,163,v,Q. Richardson 25' 3PT Jump Shot (8 PTS) (Duhon 1 AST),Made Shot,Jump Shot,0,343,21100744,2011,rg,559.000,161.000,2321.000,559,False,0.000,NaT,NaT
6853917,484,PT06M06.00S,4,1610612748,MIA,201596,Chalmers,M. Chalmers,0,0,0,NaN,0,0.000,0.000,0,v,Chalmers P.FOUL (P5.PN),Foul,Personal,0,434,21200117,2012,rg,366.000,354.000,2514.000,366,False,0.000,NaT,NaT


## Test Filter Function

In [33]:
game_id = 41500407
f_game_id = df.gameId == game_id

actionNumber = 463
f_actionNumber = df.actionNumber == actionNumber

f_isTimeout = df.actionType == "Timeout"

f_isOfficial = df.subType == "Official"

f_game_id.sum(), f_actionNumber.sum(), f_isTimeout.sum()

(np.int64(467), np.int64(17400), np.int64(320396))

In [61]:
ignore = (
    "	game_seconds_elapsed	pc_seconds	IsPlayoff	shotValue	game_date	game_date_ffill"
    + " season	season_type"
    + " videoAvailable"
    + " xLegacy	yLegacy	shotDistance"
    + " "
)
ignore_list = ignore.split()

In [62]:
df_view = UKS_NP_PD.col_filter(df, exclude=ignore_list)

Returning df without 12 columns: ['xLegacy', 'yLegacy', 'shotDistance', 'videoAvailable', 'season', 'season_type', 'game_seconds_elapsed', 'pc_seconds', 'IsPlayoff', 'shotValue', 'game_date', 'game_date_ffill']


In [63]:
UKS_NP_PD.dtt(df_view, filter=f_game_id & f_isTimeout, n=25)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,actionId,gameId,seconds_remaining,seconds_elapsed
,int64,str,int64,int64,str,int64,str,str,str,int64,float64,float64,int64,str,str,str,str,int64,int64,float64,float64
11309319,39,PT06M58.00S,1,0,NaN,1610612744,NaN,NaN,NaN,0,NaN,NaN,0,h,WARRIORS Timeout: Regular (Full 1 Short 0),Timeout,Regular,42,41500407,418.000,302.000
11309349,68,PT02M29.00S,1,0,NaN,1610612739,NaN,NaN,NaN,0,NaN,NaN,0,v,Cavaliers Timeout: Regular (Reg.1 Short 0),Timeout,Regular,72,41500407,149.000,571.000
11309425,149,PT08M24.00S,2,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,NaN,Timeout: Official,Timeout,Official,148,41500407,504.000,216.000
11309458,188,PT05M43.00S,2,0,NaN,1610612744,NaN,NaN,NaN,0,NaN,NaN,0,h,WARRIORS Timeout: Regular (Full 2 Short 0),Timeout,Regular,181,41500407,343.000,377.000
11309478,209,PT03M59.00S,2,0,NaN,1610612744,NaN,NaN,NaN,0,NaN,NaN,0,h,WARRIORS Timeout: Short (Full 2 Short 1),Timeout,Short,201,41500407,239.000,481.000
11309496,229,PT02M27.00S,2,0,NaN,1610612739,NaN,NaN,NaN,0,NaN,NaN,0,v,Cavaliers Timeout: Regular (Reg.2 Short 0),Timeout,Regular,219,41500407,147.000,573.000
11309539,274,PT08M53.00S,3,0,NaN,1610612744,NaN,NaN,NaN,0,NaN,NaN,0,h,WARRIORS Timeout: Regular (Full 3 Short 1),Timeout,Regular,262,41500407,533.000,187.000
11309557,291,PT07M38.00S,3,0,NaN,1610612739,NaN,NaN,NaN,0,NaN,NaN,0,v,Cavaliers Timeout: Regular (Reg.3 Short 0),Timeout,Regular,280,41500407,458.000,262.000
11309591,328,PT04M33.00S,3,0,NaN,1610612744,NaN,NaN,NaN,0,NaN,NaN,0,h,WARRIORS Timeout: Regular (Full 4 Short 1),Timeout,Regular,314,41500407,273.000,447.000


In [64]:
f_total_filt = f_game_id & f_isTimeout & f_isOfficial
f_total_filt.sum()

np.int64(2)

In [65]:
UKS_NP_PD.dtt(df_view, filter=f_total_filt, n=25)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,actionId,gameId,seconds_remaining,seconds_elapsed
,int64,str,int64,int64,str,int64,str,str,str,int64,float64,float64,int64,str,str,str,str,int64,int64,float64,float64
11309425,149,PT08M24.00S,2,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,NaN,Timeout: Official,Timeout,Official,148,41500407,504.000,216.000
11309707,463,PT02M50.00S,4,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,NaN,Timeout: Official,Timeout,Official,430,41500407,170.000,550.000


In [ ]:
f_nearby = UKS_NP_PD.get_nearby_rows(
    df,
    f_total_filt,
    hard_match=["period", "gameId"],
    soft_match={"seconds_elapsed": (60, "backward")},
)
f_nearby.sum()

np.int64(24)

In [71]:
UKS_NP_PD.dtt(df_view, filter=f_nearby, n=-1)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,actionId,gameId,seconds_remaining,seconds_elapsed
,int64,str,int64,int64,str,int64,str,str,str,int64,float64,float64,int64,str,str,str,str,int64,int64,float64,float64
11309418,143,PT09M20.00S,2,1610612739,CLE,2590,Williams,M. Williams,Missed,1,NaN,NaN,0,v,MISS Williams 2' Layup,Missed Shot,Layup Shot,141,41500407,560.000,160.000
11309419,144,PT09M19.00S,2,1610612744,GSW,203110,Green,D. Green,NaN,0,NaN,NaN,0,h,Green REBOUND (Off:0 Def:3),Rebound,Unknown,142,41500407,559.000,161.000
11309420,145,PT09M15.00S,2,1610612744,GSW,203110,Green,D. Green,NaN,0,NaN,NaN,0,h,Green Lost Ball Turnover (P2.T4),Turnover,Lost Ball,143,41500407,555.000,165.000
11309421,145,PT09M15.00S,2,1610612739,CLE,2544,James,L. James,NaN,0,NaN,NaN,0,v,James STEAL (1 STL),NaN,NaN,144,41500407,555.000,165.000
11309422,146,PT09M10.00S,2,1610612739,CLE,2590,Williams,M. Williams,Made,1,26.000,27.000,53,v,Williams 2' Running Layup (2 PTS) (James 4 AST),Made Shot,Running Layup Shot,145,41500407,550.000,170.000
11309423,147,PT08M49.00S,2,1610612744,GSW,203110,Green,D. Green,Made,1,29.000,27.000,56,h,Green 26' 3PT Pullup Jump Shot (10 PTS) (Iguodala 1 AST),Made Shot,Pullup Jump shot,146,41500407,529.000,191.000
11309424,148,PT08M24.00S,2,1610612739,CLE,2590,Williams,M. Williams,NaN,0,NaN,NaN,0,v,Williams Step Out of Bounds Turnover (P1.T5),Turnover,Step Out of Bounds Turnover,147,41500407,504.000,216.000
11309425,149,PT08M24.00S,2,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,NaN,Timeout: Official,Timeout,Official,148,41500407,504.000,216.000
11309426,151,PT08M24.00S,2,1610612739,CLE,202684,Thompson,T. Thompson,NaN,0,NaN,NaN,0,v,SUB: Smith FOR Thompson,Substitution,NaN,149,41500407,504.000,216.000
